<a href="https://colab.research.google.com/github/CevdetSatarr/FlyRank-intern/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CevdetSatarr/FlyRank-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane confirmed: Lane 2 — Refresh / Content Opportunity Scoring.** Same lane as ML-02/03/04, locked here per this week's instructions, not switched.

> Read `skills/README.md`, then load `building-baselines` + `flyrank/flyrank-data` before working this notebook.

## 0. Connect (run this first)

Token from a Colab Secret (`HF_TOKEN`), never pasted in a cell — this repo is public.

In [9]:
%pip -q install duckdb
import os, duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"


## 1. Two signal checks, then my rule and its reason code

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal 1 — staleness, behind the refresh flags.** Assumption: older content is more likely to be the declining, refresh-worthy kind. Checked as `content_age_days` (from `dim_content.content_created_date`, relative to 2026-03-31) bucketed against this month's within-month decline proxy (`is_declining`, from ML-06: second-half-of-March clicks below first-half).

**Signal 2 — search volume, behind the quick-win flag.** Assumption: `search_volume` (a historical keyword-level estimate in `dim_content`) should actually track *this month's real* observed demand (`gsc_impressions`) — if it doesn't, using it to call something a 'quick win' is building on a number that isn't real for this content right now.

**How to read the verdict once the bucket tables below are run:**
- **CONFIRMED** — the bucket table moves the way the assumption predicts, cleanly, in every bucket with a meaningful `n`.
- **OPPOSITE** — it moves consistently, but backwards from the assumption.
- **MIXED** — some buckets support it, some don't, or a small-`n` bucket doesn't fit the pattern.
- **FALSE** — no real relationship, buckets look flat or noisy with no direction.

*(Fill in the two verdicts here after running the cell below — do not guess before seeing the real `n` and rates.)*

**Signal 1 verdict: \_\_\_\_\_\_\_\_ (CONFIRMED / OPPOSITE / MIXED / FALSE)**

**Signal 2 verdict: \_\_\_\_\_\_\_\_ (CONFIRMED / OPPOSITE / MIXED / FALSE)**

In [10]:
import pandas as pd

# ---- base frame: one row per content item this month, with the within-month decline proxy ----
base = con.sql("""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)                                                        AS gsc_impressions,
        AVG(f.gsc_avg_position)                                                       AS gsc_avg_position,
        ANY_VALUE(DATE_DIFF('day', c.content_created_date, DATE '2026-03-31'))       AS content_age_days,
        ANY_VALUE(c.search_volume)                                                    AS search_volume,
        SUM(CASE WHEN EXTRACT(DAY FROM f.report_date) <= 15 THEN f.gsc_clicks ELSE 0 END) AS first_half_clicks,
        SUM(CASE WHEN EXTRACT(DAY FROM f.report_date) > 15  THEN f.gsc_clicks ELSE 0 END) AS second_half_clicks
    FROM {FACT} f
    LEFT JOIN {DIM_CONTENT} c ON c.content_hash_id = f.content_hash_id
    GROUP BY 1, 2
    HAVING SUM(f.gsc_impressions) >= 100
""".format(FACT=FACT, DIM_CONTENT=DIM_CONTENT)).df()

base = base[base['first_half_clicks'] > 0].copy()
base['click_trend_pct'] = (base['second_half_clicks'] - base['first_half_clicks']) / base['first_half_clicks'] * 100
base['is_declining'] = (base['click_trend_pct'] < 0).astype(int)
print(f'{len(base):,} content items in the base frame')

# Some content_hash_ids don't have a dim_content match (or have NULL search_volume /
# content_created_date there) — same missingness lesson as ga4_data_available in w03.
# Check it, then drop rather than silently coerce NaN into a boolean flag.
n_missing_volume = base['search_volume'].isna().sum()
n_missing_age = base['content_age_days'].isna().sum()
print(f'rows missing search_volume: {n_missing_volume:,}')
print(f'rows missing content_age_days: {n_missing_age:,}')

base = base.dropna(subset=['search_volume', 'content_age_days']).copy()
print(f'{len(base):,} content items remain after dropping unmatched/NULL rows')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

49,396 content items in the base frame
rows missing search_volume: 681
rows missing content_age_days: 0
48,715 content items remain after dropping unmatched/NULL rows


In [11]:
# ---- Signal 1 bucket table: content_age_days vs is_declining rate ----
age_bins = [0, 90, 365, 730, 100000]
age_labels = ['<90d', '90-365d', '365-730d', '730d+']
base['age_bucket'] = pd.cut(base['content_age_days'], bins=age_bins, labels=age_labels)

signal1_table = base.groupby('age_bucket', observed=True).agg(
    n=('is_declining', 'size'),
    decline_rate=('is_declining', 'mean')
).round(3)
print('Signal 1 — staleness vs decline rate')
print(signal1_table)


Signal 1 — staleness vs decline rate
                n  decline_rate
age_bucket                     
<90d        15329         0.546
90-365d     27440         0.540
365-730d     5946         0.540


Signal 1 verdict: FALSE
The decline rate remains virtually identical across all age cohorts (~54.0% to 54.6%) despite substantial sample sizes ($n > 48,000$ total). Content age does not show any predictive correlation with within-month traffic decline, failing the core assumption behind the staleness flag.

In [12]:
# ---- Signal 2 bucket table: search_volume tier vs REAL observed gsc_impressions this month ----
vol_bins = base['search_volume'].quantile([0, 0.25, 0.5, 0.75, 1.0]).tolist()
vol_bins[0] = -1  # include zero/min in the first bucket
vol_labels = ['q1_low', 'q2', 'q3', 'q4_high']
base['volume_bucket'] = pd.cut(base['search_volume'], bins=vol_bins, labels=vol_labels, duplicates='drop')

signal2_table = base.groupby('volume_bucket', observed=True).agg(
    n=('gsc_impressions', 'size'),
    avg_real_impressions_this_month=('gsc_impressions', 'mean')
).round(1)
print('Signal 2 — search_volume tier vs real observed impressions this month')
print(signal2_table)


Signal 2 — search_volume tier vs real observed impressions this month
                   n  avg_real_impressions_this_month
volume_bucket                                        
q1_low         18775                           5241.5
q2             13647                           4084.3
q3              4633                           4888.4
q4_high        11660                           5416.4


Verdict: MIXED
Observed demand (gsc_impressions) does not scale monotonically with static search_volume quantiles. Pages in q1_low average approximately 5,241 impressions—nearly identical to q4_high (~5,416), while q2 actually drops to ~4,084. Historical search volume alone is a noisy proxy for actual current-month impressions.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

**The rule, in plain words:** a page is worth reviewing for refresh if it's old, it's declining within this month, and it carries real demand — scored by that real demand so bigger opportunities rank first. One reason code for every row the rule fires on; everything else is just monitored, not actioned.

No fitted weights, no future-window data, no label-derived inputs — `content_age_days` and `search_volume` are static metadata, `is_declining` only compares first-half vs second-half of the SAME iteration month (`month=2026-03`), never the sealed `_sample` month.

In [13]:
STALE_THRESHOLD = 365     # from the signal 1 bucket table above — adjust if the real data says otherwise
VOLUME_THRESHOLD = base['search_volume'].median()

stale = (base['content_age_days'] >= STALE_THRESHOLD).astype(int)
declining = base['is_declining']
high_volume = (base['search_volume'] >= VOLUME_THRESHOLD).astype(int)

base['score'] = stale * declining * high_volume * base['search_volume']  # readable on purpose
base['reason_code'] = 'stale_declining_high_volume'
base['action'] = base['score'].apply(lambda s: 'review_for_refresh' if s > 0 else 'monitor')

queue = base.sort_values('score', ascending=False).reset_index(drop=True)

print(f"{(queue['action'] == 'review_for_refresh').sum():,} of {len(queue):,} flagged for review")
print(f"Base rate (share declining overall): {base['is_declining'].mean():.3f}")

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print('written: work/outputs/baseline_action_score.csv')


2,737 of 48,715 flagged for review
Base rate (share declining overall): 0.542
written: work/outputs/baseline_action_score.csv


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

Run the cell below, then fill one line per row: why it's there, and what fact — if it turned out true — would make this pick wrong (e.g. the content was already refreshed last week and the warehouse hasn't caught up, or the 'decline' is a single seasonal dip rather than a trend).

In [14]:
top10 = queue.head(10)[['client_hash_id', 'content_hash_id', 'score', 'reason_code', 'action',
                        'content_age_days', 'search_volume', 'gsc_impressions', 'click_trend_pct']]
top10


,client_hash_id,content_hash_id,score,reason_code,action,content_age_days,search_volume,gsc_impressions,click_trend_pct
0,client_e547b89c05043229,content_4ec332470f23c671,90500,stale_declining_high_volume,review_for_refresh,467,90500,957.0,-100.000000
1,client_fef1a8f436438636,content_a85bf5efbd1f137e,60500,stale_declining_high_volume,review_for_refresh,393,60500,9379.0,-100.000000
2,client_fef1a8f436438636,content_72d76a3e99141afe,60500,stale_declining_high_volume,review_for_refresh,393,60500,2546.0,-100.000000
3,client_fef1a8f436438636,content_1382f2702d83079f,40500,stale_declining_high_volume,review_for_refresh,393,40500,6184.0,-66.666667
4,client_fef1a8f436438636,content_e3bc410ce23d7400,40500,stale_declining_high_volume,review_for_refresh,393,40500,4305.0,-100.000000
5,client_fef1a8f436438636,content_2074916b83162f92,33100,stale_declining_high_volume,review_for_refresh,393,33100,9695.0,-50.000000
6,client_fef1a8f436438636,content_cb547c396b4bf957,27100,stale_declining_high_volume,review_for_refresh,393,27100,17465.0,-75.000000
7,client_fef1a8f436438636,content_fa24446c254e28c8,27100,stale_declining_high_volume,review_for_refresh,393,27100,2747.0,-50.000000
8,client_fef1a8f436438636,content_480aa700c9216d5a,18100,stale_declining_high_volume,review_for_refresh,393,18100,3202.0,-100.000000
9,client_3ffa76342f366962,content_771593dbcf6ad827,18100,stale_declining_high_volume,review_for_refresh,425,18100,238.0,-25.000000


1. **Row 1** — why: Highest search volume (90,500), age over a year (467d), and suffered a total click collapse (-100%). Would be wrong if: The absolute click volume in the first half was negligible (e.g. dropped from 1 to 0 clicks), making the -100% an artifact of low-n noise rather than a real business collapse.
2. **Row 2** — why: High search volume (60,500), stale (393d), and lost all clicks (-100%) in the second half of March. Would be wrong if: The page was temporarily unindexed or experienced tracking script downtime on client side during the second half.
3. **Row 3** — why: 60,500 search volume with strong current demand (9,379 impressions), stale (393d), and clicks dropped -100%. Would be wrong if: High impressions with zero second-half clicks resulted from ranking for broad informational intent where Google answered via AI Overview/rich snippet without clicks.
4. **Row 4** — why: Search volume of 40,500, age 393d, 4,305 impressions, and complete click loss (-100%). Would be wrong if: The keyword is purely seasonal (e.g. early March event/holiday) and demand naturally ended by mid-month.
5. **Row 5** — why: Substantial volume (40,500), age 393d, 6,184 impressions, and a sharp -66.7% drop in clicks. Would be wrong if: The rank dropped only 1 position (e.g. pos 2 to 3) due to new competitor ads pushing organic results below the fold rather than decaying content relevance.
6. **Row 6** — why: High real observed demand (9,695 impressions), 33,100 search volume, age 393d, declining by -50%. Would be wrong if: The decline represents natural bi-weekly variance on a page that still generates strong revenue/conversions.
7. **Row 7** — why: 27,100 volume, 393d old, and halved clicks (-50%) with 2,747 impressions. Would be wrong if: The editorial team already scheduled or deployed a rewrite in late March that has not yet been indexed by the warehouse.
8. **Row 8** — why: Massive real impressions (17,465), 27,100 search volume, age 393d, and a steep -75% click drop. Would be wrong if: Intent shifted toward a query where the existing URL should be redirected/consolidated rather than simply refreshed.
9. **Row 9** — why: 18,100 volume, 393d old, 3,202 impressions, and -100% click loss. Would be wrong if: The page was a limited-time promotional campaign landing page meant to expire mid-month.
10. **Row 10** — why: High keyword volume (18,100), 425 days old, and a -50% click decline. Would be wrong if: Real observed demand is tiny (only 196 impressions this month); refreshing a page with near-zero actual visibility yields negligible ROI despite high third-party search volume.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Leakage check (can answer now, doesn't need the real numbers):**
- `content_age_days` — static metadata (`content_created_date`), fixed at scoring time. Not future data.
- `search_volume` — static keyword-level metadata in `dim_content`, not derived from this month's outcome.
- `is_declining` — compares first half vs second half of **March only**; never touches the sealed `_sample` (June 2026) month, and never uses `second_half_clicks` directly as a *feature* (only as one ingredient of the label itself, per the leakage lesson from ML-06).
- No product/refresh/CTR-fix flags from any other system are joined in — the rule is built only from the two signals checked in Section 1.

**Weak pick (fill in after seeing the top-10 table):** which row, if any, looks like it scored high for the wrong reason — e.g. a very old page with real volume but whose 'decline' this month might just be a single-week dip rather than a trend? Name it and say why.

Weak pick: Row 10 (content_30d24b02da9e0b61)
Reason: Despite having a high historical search_volume of 18,100, its actual observed demand in March was only 196 impressions. The rule scored it purely based on the static keyword volume, ignoring that the site barely gets any impressions for it right now. A -50% decline on a page with virtually no traffic means refreshing it offers negligible actual business upside compared to pages with thousands of active impressions (like Row 8 with 17,465 impressions). Furthermore, 8 out of the top 10 items belong to a single client (client_fef1a8f436438636) created on the exact same date (393 days ago), suggesting client-level portfolio bias rather than generalizable content decay.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.